# Member C 蛋白特征 MLP

使用时间、评分类型和当前蛋白 NPX 预测 UPDRS。`imputation_method` 可选 `"median"` 或 `"iterative"`；后者用 BayesianRidge 迭代预测缺失蛋白。填补模型只在训练部分的唯一访问上拟合，不接触验证患者或 UPDRS 标签。

`max_missing_rate=1.0` 保留全部蛋白，`use_proteins=False` 使用时间参照。修改配置后从头运行。

In [1]:
import json
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler
from torch import nn
import sys

root = Path.cwd()
if not (root / "common").is_dir():
    root = root.parent

sys.path.insert(0, str(root))
from common import SEED, TARGETS
from common.data import load_raw
from common.experiments import log_experiment
from common.interface import model_folds
from common.metric import evaluate_oof, smape_plus_one
from common.reproducibility import set_seed


In [2]:
use_proteins = True
imputation_method = "iterative"  # "median" 或 "iterative"
max_missing_rate = 1.0  # 1.0 保留全部蛋白，0.0 仅保留训练部分完全无缺失的蛋白
epochs = 200
learning_rate = 0.01
weight_decay = 0.01
set_seed(SEED)
torch.manual_seed(SEED)

shared = root / "artifacts/shared"
output_dir = root / "artifacts/member_c"
raw = load_raw(root / "amp-parkinsons-disease-progression-prediction")
samples = pd.read_csv(shared / "samples.csv")
patients = pd.read_csv(shared / "patient_folds.csv")
manifest = json.loads((shared / "manifest.json").read_text())


In [3]:
def features(frame, protein_columns):
    time = frame[["visit_month", "horizon", "target_month"]].to_numpy()
    target = np.column_stack([frame.target.eq(name) for name in TARGETS])
    proteins = frame[protein_columns].to_numpy()
    return np.column_stack([time, target, proteins]).astype(np.float32)


In [4]:
def prepare_features(train_frame, valid_frame, protein_columns, method):
    train_values = features(train_frame, protein_columns)
    valid_values = features(valid_frame, protein_columns)
    if not protein_columns:
        return train_values, valid_values

    # One measurement per visit: forecast horizons/targets must not duplicate it.
    unique_proteins = train_frame.drop_duplicates("visit_id")[protein_columns]
    if method == "median":
        imputer = SimpleImputer(strategy="median", keep_empty_features=True)
        imputer.fit(unique_proteins.to_numpy())
        train_values[:, 7:] = imputer.transform(train_values[:, 7:])
        valid_values[:, 7:] = imputer.transform(valid_values[:, 7:])
    elif method == "iterative":
        # StandardScaler preserves NaN; scaling avoids abundance magnitude
        # dominating the regularized protein-to-protein regressions.
        protein_scaler = StandardScaler()
        scaled_proteins = protein_scaler.fit_transform(unique_proteins.to_numpy())
        imputer = IterativeImputer(
            estimator=BayesianRidge(), initial_strategy="median",
            max_iter=10, tol=0.01, n_nearest_features=min(20, len(protein_columns)),
            skip_complete=True, keep_empty_features=True, random_state=SEED,
        )
        imputer.fit(scaled_proteins)
        for values in (train_values, valid_values):
            observed = values[:, 7:].copy()
            filled = imputer.transform(protein_scaler.transform(observed))
            filled = protein_scaler.inverse_transform(filled).clip(min=0)
            # Preserve every measured NPX exactly; replace only missing entries.
            values[:, 7:] = np.where(np.isnan(observed), filled, observed)
    else:
        raise ValueError("method must be 'median' or 'iterative'")
    return train_values, valid_values


In [5]:
class MLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.network(x).squeeze(1)


In [6]:
def train_model(model, x_train, y_train, epochs, learning_rate, weight_decay):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=learning_rate, weight_decay=weight_decay,
    )
    loss_fn = nn.MSELoss()
    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(x_train), y_train)
        loss.backward()
        optimizer.step()
    return model


In [7]:
started = perf_counter()
predictions = []
proteins_by_fold = {}

for data in model_folds(raw, samples, patients):
    # Freeze protein columns from the training fold, including their order.
    protein_columns = [
        col for col in data.train_features if col.startswith("protein::")
    ] if use_proteins else []
    # Count each training visit once, not once per forecast request.
    train_visits = data.train_features.drop_duplicates("visit_id")
    missing_rate = train_visits[protein_columns].isna().mean()
    protein_columns = missing_rate.index[missing_rate <= max_missing_rate].tolist()
    proteins_by_fold[data.fold] = protein_columns
    train_values, valid_values = prepare_features(
        data.train_features, data.valid_features, protein_columns, imputation_method,
    )
    scaler = StandardScaler()
    x_train = torch.from_numpy(scaler.fit_transform(train_values))
    x_valid = torch.from_numpy(scaler.transform(valid_values))
    y_train = torch.tensor(data.train_samples.rating.to_numpy(), dtype=torch.float32)

    model = MLP(input_size=x_train.shape[1])
    model = train_model(model, x_train, y_train, epochs, learning_rate, weight_decay)

    model.eval()
    with torch.no_grad():
        rating = model(x_valid).clamp_min(0).numpy()
    predictions.append(pd.DataFrame({
        "prediction_id": data.valid_samples.prediction_id.to_numpy(), "rating": rating,
    }))
    score = smape_plus_one(data.valid_samples.rating, rating)
    print(f"fold {data.fold}: proteins={len(protein_columns)}, SMAPE+1={score:.4f}", flush=True)

runtime_seconds = perf_counter() - started


fold 0: proteins=227, SMAPE+1=80.0747
fold 1: proteins=227, SMAPE+1=84.6628
fold 2: proteins=227, SMAPE+1=86.6417
fold 3: proteins=227, SMAPE+1=85.4352
fold 4: proteins=227, SMAPE+1=86.0696


In [8]:
oof = pd.concat(predictions, ignore_index=True)
evaluation = evaluate_oof(samples, oof)
pd.Series(evaluation)


fold_scores      {0: 80.07473601941814, 1: 84.66280985237846, 2...
mean_score                                               84.576814
std_score                                                 2.345426
oof_score                                                84.474447
n_predictions                                                12697
dtype: object

In [9]:
branch = f"protein_{imputation_method}_uniquevisits_missing{max_missing_rate:g}" if use_proteins else "time_only"
experiment_id = f"memberC_mlp_{branch}_" + pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
output_dir.mkdir(parents=True, exist_ok=True)
oof.to_csv(output_dir / f"{experiment_id}_oof.csv", index=False)
log_experiment(
    output_dir / "experiment_results.csv",
    experiment_id=experiment_id,
    model="PyTorch MLP input-32-16-1",
    features=f"time,target_one_hot; proteins={use_proteins}, max_missing_rate={max_missing_rate}; no peptides or supplemental",
    evaluation=evaluation,
    seed=SEED,
    runtime_seconds=runtime_seconds,
    split_id=manifest["split_id"],
    config={"epochs": epochs, "lr": learning_rate, "weight_decay": weight_decay,
            "loss": "MSE", "optimizer": "AdamW", "batch": "full", "device": "cpu",
            "use_proteins": use_proteins, "max_missing_rate": max_missing_rate,
            "imputation": imputation_method, "imputer_fit_unit": "unique training visit",
            "iterative_parameters": {"estimator": "BayesianRidge", "max_iter": 10,
                                     "n_nearest_features": 20, "tol": 0.01, "random_state": SEED}
                if imputation_method == "iterative" else {}, "proteins_by_fold": proteins_by_fold,
            "torch_version": torch.__version__},
    notes="Protein selection by train-fold unique-visit missing rate; imputer fit on unique training visits; iterative model uses standardized proteins only; final train-fold StandardScaler; raw NPX; predictions clipped at zero.",
)
print(output_dir)


/Users/willciao/Documents/College/CS4486AI/Projects/4486/artifacts/member_c


## 蛋白缺失率阈值对照

固定 seed、200 轮、MSE、AdamW；仅改变训练折内的蛋白筛选阈值。分数越低越好。

| 最大缺失率 | 各折保留蛋白数 | 五折平均 SMAPE+1 |
|---|---:|---:|
| 仅时间特征 | 0 | 71.0994 |
| 0% | 27–76 | 76.2922 |
| 1% | 109–116 | 84.5998 |
| 5% | 148–151 | 81.6627 |
| 10% | 166–168 | 83.8067 |
| 20% | 189–192 | 81.7441 |
| 100% | 227–227 | 84.4344 |

筛除缺失较多的蛋白有时优于保留全部蛋白，但本轮所有蛋白版本均未超过时间特征参照。筛选只移除蛋白列，不删除访问或预测请求；即使训练部分缺失率为零，验证部分仍可能缺失，因此仍需填充。

这是单个 seed 下的探索性阈值比较，不能将挑选出的最好分数视为独立测试结果。本表为此前的阈值实验；本次默认配置为全部蛋白和迭代填补。此前中位数在预测请求行上拟合，本次填补器统一按唯一训练访问拟合，因此另重跑中位数对照。

## 模型填补与中位数对照

本次都保留全部 227 种蛋白，并在唯一训练访问上拟合填补器；同一 seed、200 轮、相同下游 MLP。此前的中位数实验按预测请求行计算统计量，因此以本次重跑的中位数为直接对照。

| 填补方法 | 五折平均 SMAPE+1 | 五折标准差 |
|---|---:|---:|
| median | 84.4233 | 3.6516 |
| iterative | 84.5768 | 2.3454 |

本次迭代填补未改善 UPDRS 预测。五折均出现达到 max_iter=10 但未满足 tol=0.01 的收敛提示，因此结果仅代表此固定配置，不能推断所有模型填补方法均无效。单个 seed 的微小差异不应视为稳定结论。

验证检查：12,697 条 OOF 请求完整且唯一；处理后的输入均为有限数值；检查的训练/验证输入中所有原始可观测特征保持不变。

本次运行设置 OPENBLAS_NUM_THREADS=1，避免小矩阵多线程开销。此设置影响速度，不是强制确定性运算；未写入训练单元格。
